In [69]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.ticker import PercentFormatter

import utils
import utils_plot
import utils_print

sns.set_context("talk")

try:
    sql_conn.close()
except Exception:
    pass
sql_conn = utils.SQLConnection()

In [87]:
df = sql_conn.run_sql_file("cohortes.sql")

In [93]:
df

,CohortQuarter,MonthsAfterFirstOrder,CustomerType,ActiveCustomers,RetentionRate,TotalSpending,AvgOrderValue,CohortSize,TotalOrders,OrdersWithReason,MostCommonReason,MostCommonReasonPct
0,2011-Q2,0,B2C,146,100.00,9.184928e+05,3733.7104,146,146,100,Manufacturer,68.49
1,2011-Q2,23,B2C,1,0.68,5.160284e+03,2580.1419,146,1,1,On Promotion,100.00
2,2011-Q2,24,B2C,15,10.27,4.656121e+04,2586.7338,146,15,14,Price,73.33
3,2011-Q2,25,B2C,5,3.42,1.134530e+04,1620.7572,146,5,4,Price,80.00
4,2011-Q2,26,B2C,12,8.22,2.101456e+04,1751.2132,146,12,4,Price,25.00
...,...,...,...,...,...,...,...,...,...,...,...,...
214,2014-Q1,4,B2C,61,1.93,1.843916e+04,288.1118,3164,62,59,Price,79.03
215,2014-Q1,5,B2C,23,0.73,1.896281e+03,75.8512,3164,23,23,Price,95.65
216,2014-Q2,0,B2C,2854,100.00,1.960194e+06,609.3235,2854,2885,2618,Price,82.18
217,2014-Q2,1,B2C,41,1.44,3.298305e+03,68.7146,2854,43,41,Price,83.72


In [89]:
# Convert 'RetentionRate' and 'MostCommonReasonPct' to float in b2c_data (no division by 100)
df['RetentionRate'] = df['RetentionRate'].str.rstrip('%').astype(float)
df['MostCommonReasonPct'] = df['MostCommonReasonPct'].str.rstrip('%').astype(float)

In [ ]:
df.info()

In [133]:
b2c_data = df[df['CustomerType'] == 'B2C']
b2c_data = b2c_data[b2c_data['MonthsAfterFirstOrder'] > 0]

plt.figure(figsize=(14, 8))
sns.set_context("talk", font_scale=1.2)

sns.lineplot(
    data=b2c_data,
    x='MonthsAfterFirstOrder',
    y='RetentionRate',
    hue='CohortQuarter',
    palette='tab20',
    # style='CohortQuarter',
    marker='o',
    linewidth=2.5,
    markersize=8
)

plt.grid(True, which='both', axis='both', alpha=0.7)

plt.title('Retención de clientes por cohorte', fontsize=15, pad=10)
plt.xlabel('Meses desde la primera compra', fontsize=15)
plt.ylabel('Tasa de retención', fontsize=15)
plt.legend(title='Cohorte', loc='best', bbox_to_anchor=(1.18, 1), fontsize=13, title_fontsize=15)

plt.tight_layout()
save_fp = os.path.join(utils.PLOTS_DIRPATH_, "coohortes1.png")
print(f'Figura guardada en "{save_fp}".')
plt.savefig(save_fp)
# plt.show()
plt.close()

Figura guardada en "/home/pc/P/M/MR/tp/fuente/relational-models/TP/graficos/coohortes1.png".


In [151]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

sns.set_context("talk", font_scale=1.2)

b2c_data = df[df['CustomerType'] == 'B2C']

heatmap_data = b2c_data.pivot_table(
    index='CohortQuarter',
    columns='MonthsAfterFirstOrder',
    values='MostCommonReason',
    aggfunc='first'
)

plt.figure(figsize=(15, 6))

STATE_COLORS = {
    'Sin datos': 'white',
    'None': '#CCCCCC' 
}

unique_reasons = [r for r in heatmap_data.stack().unique() 
                 if r is not None and not pd.isna(r)]


for i, reason in enumerate(unique_reasons):
    STATE_COLORS[reason] = plt.cm.tab20(i % 20)  # Cycle through tab20 colors

state_mapping = {state: i for i, state in enumerate(STATE_COLORS)}
inverse_mapping = {i: state for state, i in state_mapping.items()}

def map_state(value):
    if pd.isna(value):
        return state_mapping['Sin datos']
    elif value is None:
        return state_mapping['None']
    return state_mapping.get(value, state_mapping['Sin datos'])

mapped_data = heatmap_data.applymap(map_state)

colors = [STATE_COLORS[state] for state in inverse_mapping.values()]
cmap = plt.cm.colors.ListedColormap(colors)

sns.heatmap(
    mapped_data,
    cmap=cmap,
    cbar=False,
    linewidth=0.5,
    linecolor='lightgray',
    square=True,
    vmin=0,
    vmax=len(STATE_COLORS)-1
)

legend_patches = [plt.Rectangle((0,0),1,1, color=color, label=state) 
                 for state, color in STATE_COLORS.items()]

plt.legend(handles=legend_patches,
           bbox_to_anchor=(1.02, 1),
           loc='upper left',
           title='Razones',
           fontsize=12,
           title_fontsize=14)

plt.title('Razones de venta por cohorte y período de tiempo', fontsize=15)
plt.xlabel('Meses desde la primera compra', fontsize=15)
plt.ylabel('Cohorte', fontsize=15)

plt.xticks(fontsize=12, rotation=0, ha='right')
plt.yticks(fontsize=12)

plt.tight_layout()
save_fp = os.path.join(utils.PLOTS_DIRPATH_, "coohortes2.png")
print(f'Figura guardada en "{save_fp}".')
plt.savefig(save_fp)
# plt.show()
plt.close()

Figura guardada en "/home/pc/P/M/MR/tp/fuente/relational-models/TP/graficos/coohortes2.png".
